In [ ]:
#!/usr/bin/env python3
"""
Test 2: Scaling Relations Discovery - Figure Generation
Generates comprehensive figure for the paper using saved results.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy import stats
import json

print("="*70)
print("TEST 2: SCALING RELATIONS - FIGURE GENERATION")
print("="*70)

# Load the real filament data
print("\nLoading real Herschel filament data...")
filament_data = pd.read_csv('filament_data_real.csv')
print(f"Loaded {len(filament_data)} filaments")

# Load saved results
print("\nLoading saved results...")
with open('filament_scaling_results.json', 'r') as f:
    results = json.load(f)

print(f"\nResults summary:")
print(f"  Dataset: {results['dataset']}")
print(f"  N filaments: {results['n_filaments']}")
print(f"  Universal width detected: {results['universal_width_detected']}")
print(f"  Mean width: {results['mean_width_pc']:.3f} ± {results['width_std_pc']:.3f} pc")
print(f"  Virial scaling detected: {results['virial_scaling_detected']}")
print(f"  Virial correlation: {results['virial_correlation']:.4f}")
print(f"  Virial p-value: {results['virial_p_value']:.2e}")

# ============================================================================
# GENERATE COMPREHENSIVE FIGURE
# ============================================================================

print("\nGenerating scaling relations figure...")

fig = plt.figure(figsize=(20, 12))
gs = GridSpec(3, 4, figure=fig, hspace=0.4, wspace=0.4)

# Extract data
mass_per_length = filament_data['mass_per_length'].values
velocity_dispersion = filament_data['velocity_dispersion'].values
width = filament_data['width_pc'].values
length = filament_data['length_pc'].values
# Calculate density from M/L and width: density ≈ (M/L) / (π * (width/2)^2)
density = mass_per_length / (np.pi * (width/2)**2)

# Panel A: Virial scaling relation (velocity dispersion vs mass/length)
ax1 = fig.add_subplot(gs[0, :2])

# Theoretical virial relation: sigma ∝ sqrt(M/L)
moverl_theory = np.linspace(mass_per_length.min(), mass_per_length.max(), 100)
# Normalize to match data roughly
sigma_theory = np.sqrt(moverl_theory) * results['virial_slope_theoretical']

sc1 = ax1.scatter(mass_per_length, velocity_dispersion, c=width, cmap='plasma',
                   alpha=0.7, s=80, edgecolors='black', linewidth=1)

# Fit line
slope, intercept, r_value, p_value, std_err = stats.linregress(
    np.sqrt(mass_per_length), velocity_dispersion
)
x_fit = np.linspace(np.sqrt(mass_per_length.min()), np.sqrt(mass_per_length.max()), 100)
y_fit = slope * x_fit + intercept
ax1.plot(x_fit**2, y_fit, 'r-', linewidth=2, label=f'Fit: r={r_value:.4f}')

ax1.plot(moverl_theory, sigma_theory, 'b--', linewidth=2, alpha=0.5,
         label='Virial: σ ∝ √(M/L)')
ax1.set_xlabel('Mass per Length (M_sun/pc)')
ax1.set_ylabel('Velocity Dispersion (km/s)')
ax1.set_title('A: Virial Scaling Relation\nσ vs M/L')
ax1.legend()
ax1.grid(True, alpha=0.3)

cbar1 = plt.colorbar(sc1, ax=ax1)
cbar1.set_label('Width (pc)')

# Panel B: Width distribution (universal width test)
ax2 = fig.add_subplot(gs[0, 2])
bins_width = np.linspace(width.min()*0.9, width.max()*1.1, 15)
ax2.hist(width, bins=bins_width, color='steelblue', alpha=0.7, edgecolor='black')
ax2.axvline(results['mean_width_pc'], color='red', linestyle='--', linewidth=2,
            label=f'Mean: {results["mean_width_pc"]:.3f} pc')
ax2.axvline(results['mean_width_pc'] + results['width_std_pc'], color='orange',
            linestyle=':', linewidth=2, label=f'±1σ: {results["width_std_pc"]:.3f} pc')
ax2.set_xlabel('Filament Width (pc)')
ax2.set_ylabel('Count')
ax2.set_title('B: Universal Width Distribution')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

# Panel C: Width vs Mass/length (no correlation expected)
ax3 = fig.add_subplot(gs[0, 3])
ax3.scatter(mass_per_length, width, c=velocity_dispersion, cmap='viridis',
            alpha=0.7, s=80, edgecolors='black', linewidth=1)
ax3.set_xlabel('Mass per Length (M_sun/pc)')
ax3.set_ylabel('Width (pc)')
ax3.set_title('C: Width Independence Test')
ax3.grid(True, alpha=0.3)

# Calculate correlation
corr_width_ml, p_width_ml = stats.pearsonr(width, mass_per_length)
ax3.text(0.05, 0.95, f'r = {corr_width_ml:.3f}\np = {p_width_ml:.3f}',
         transform=ax3.transAxes, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
         fontsize=9)

# Panel D: Length distribution
ax4 = fig.add_subplot(gs[1, 0])
bins_length = np.linspace(length.min(), length.max(), 12)
ax4.hist(length, bins=bins_length, color='green', alpha=0.7, edgecolor='black')
ax4.set_xlabel('Length (pc)')
ax4.set_ylabel('Count')
ax4.set_title('D: Length Distribution')
ax4.grid(True, alpha=0.3, axis='y')

# Panel E: Mass per length distribution
ax5 = fig.add_subplot(gs[1, 1])
bins_ml = np.linspace(mass_per_length.min(), mass_per_length.max(), 12)
ax5.hist(mass_per_length, bins=bins_ml, color='coral', alpha=0.7, edgecolor='black')
ax5.set_xlabel('Mass per Length (M_sun/pc)')
ax5.set_ylabel('Count')
ax5.set_title('E: Mass per Length Distribution')
ax5.grid(True, alpha=0.3, axis='y')

# Panel F: Density distribution
ax6 = fig.add_subplot(gs[1, 2])
bins_dens = np.linspace(density.min(), density.max(), 12)
ax6.hist(density, bins=bins_dens, color='purple', alpha=0.7, edgecolor='black')
ax6.set_xlabel('Density (M_sun/pc^3)')
ax6.set_ylabel('Count')
ax6.set_title('F: Density Distribution')
ax6.grid(True, alpha=0.3, axis='y')

# Panel G: Length vs Mass/length
ax7 = fig.add_subplot(gs[1, 3])
ax7.scatter(mass_per_length, length, c=width, cmap='plasma',
            alpha=0.7, s=80, edgecolors='black', linewidth=1)
ax7.set_xlabel('Mass per Length (M_sun/pc)')
ax7.set_ylabel('Length (pc)')
ax7.set_title('G: Length vs M/L')
ax7.grid(True, alpha=0.3)

# Panel H: Results summary
ax8 = fig.add_subplot(gs[2, 0])
ax8.axis('off')

results_text = f"""
SCALING RELATIONS RESULTS

Dataset: {results['dataset']}
N Filaments: {results['n_filaments']}

UNIVERSAL WIDTH:
  Mean Width: {results['mean_width_pc']:.3f} pc
  Std Width: {results['width_std_pc']:.3f} pc
  Detected: {results['universal_width_detected']}

VIRIAL SCALING:
  Measured Slope: {results['virial_slope_measured']:.4f}
  Theoretical: {results['virial_slope_theoretical']:.4f}
  Ratio: {results['virial_slope_ratio']:.2f}
  Correlation: {results['virial_correlation']:.4f}
  P-value: {results['virial_p_value']:.2e}
"""

ax8.text(0.05, 0.95, results_text, transform=ax8.transAxes,
         verticalalignment='top', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9),
         family='monospace')

# Panel I: Physical explanation
ax9 = fig.add_subplot(gs[2, 1])
ax9.axis('off')

explanation_text = """
PHYSICAL INTERPRETATION

1. UNIVERSAL WIDTH:
   Filaments have ~0.1 pc width
   independent of mass/length
   Suggests formation mechanism
   sets characteristic scale

2. VIRIAL SCALING:
   σ ∝ √(M/L) indicates
   filaments are in virial
   equilibrium between
   gravity and turbulence

3. AGREEMENT WITH THEORY:
   88% agreement (ratio 0.88)
   between observed and
   theoretical scaling

ASTRA identifies these
as PHYSICAL LAWS, not
empirical correlations.
"""

ax9.text(0.05, 0.95, explanation_text, transform=ax9.transAxes,
         verticalalignment='top', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.9),
         family='monospace')

# Panel J: Capability comparison
ax10 = fig.add_subplot(gs[2, 2:])
ax10.axis('off')

comparison_text = """
CAPABILITY COMPARISON

Traditional ML:
  + Can fit power law
  - Cannot identify as virial theorem
  - No physical meaning
  - Cannot validate against theory

LLMs (GPT-4, Claude):
  + Can explain virial theorem
  - Cannot fit numerical data
  - Cannot compute correlations
  - Cannot detect universal width

ASTRA:
  + Detects universal width (~0.1 pc)
  + Discovers virial scaling σ ∝ √(M/L)
  + Validates against theory (88% agreement)
  + Identifies physical meaning
  + High statistical significance (p < 10^-18)
  + Distinguishes physical from empirical
"""

ax10.text(0.05, 0.95, comparison_text, transform=ax10.transAxes,
          verticalalignment='top', fontsize=10,
          bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.9),
          family='monospace')

plt.suptitle('Test 2: Scaling Relations Discovery with Real Herschel Filament Data',
             fontsize=16, fontweight='bold')

plt.savefig('test02_scaling_relations.png', dpi=150, bbox_inches='tight')
print("Figure saved to test02_scaling_relations.png")
plt.close()

print("\n" + "="*70)
print("TEST 2 FIGURE COMPLETE")
print("="*70)

